# EM Algorithm

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binom

def estimate_XY_em(counts, N, n_init=10, max_iter=500, tol=1e-8, seed=0):
    counts = np.asarray(counts, dtype=float)
    M = len(counts)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        Y = rng.uniform(0.01, 0.3)
        X = rng.uniform(0.5, 0.99)
        pi = rng.uniform(0.1, 0.5)
        prev_ll = -np.inf
        for _ in range(max_iter):
            log_pX = binom.logpmf(counts, N, np.clip(X, 1e-9, 1-1e-9))
            log_pY = binom.logpmf(counts, N, np.clip(Y, 1e-9, 1-1e-9))
            log_a = np.log(pi + 1e-300) + log_pX
            log_b = np.log(1 - pi + 1e-300) + log_pY
            m = np.maximum(log_a, log_b)
            denom = m + np.log(np.exp(log_a - m) + np.exp(log_b - m))
            resp = np.exp(log_a - denom)
            ll = np.sum(denom)

            s1 = resp.sum()
            s0 = (1 - resp).sum()
            X = (resp * counts).sum() / (N * s1 + 1e-12)
            Y = ((1 - resp) * counts).sum() / (N * s0 + 1e-12)
            pi = s1 / M
            X = np.clip(X, 1e-6, 1 - 1e-6)
            Y = np.clip(Y, 1e-6, 1 - 1e-6)
            pi = np.clip(pi, 1e-6, 1 - 1e-6)
            if abs(ll - prev_ll) < tol:
                break
            prev_ll = ll

        if X < Y:
            X, Y = Y, X
            pi = 1 - pi
            resp = 1 - resp

        if best is None or ll > best['loglik']:
            best = dict(X=X, Y=Y, pi=pi, resp=resp, loglik=ll)

    return best

## TC

In [2]:
df = pd.read_csv(r'Boruta-Tc.csv')
counts = (df.iloc[:, :] <= 1).sum(axis=0).values
M = len(df.columns)
res = estimate_XY_em(counts, 20, n_init=100, max_iter=1000)
print(f'Estimated X: {res["X"]:.4f}, Y: {res["Y"]:.4f}, pi: {res["pi"]:.4f}, log-likelihood: {res["loglik"]:.4f}')
df_ = pd.DataFrame({
    'feature': df.columns,
    'count': counts.astype(int),
    'prob_important': res['resp']
})
df_ = df_[df_['prob_important'] >= 0.95]
df_.reset_index(drop=True, inplace=True)
df_

Estimated X: 0.9293, Y: 0.0018, pi: 0.0651, log-likelihood: -866.9602


,feature,count,prob_important
0,RDKit2D_MinEStateIndex,20,1.0
1,RDKit2D_qed,20,1.0
2,RDKit2D_HeavyAtomMolWt,20,1.0
3,RDKit2D_HallKierAlpha,10,1.0
4,RDKit2D_LabuteASA,20,1.0
...,...,...,...
85,ChemInfo_ISO,20,1.0
86,ChemInfo_HOMO,20,1.0
87,ChemInfo_GTOT,20,1.0
88,ChemInfo_HARD,15,1.0


## PC

In [3]:
df = pd.read_csv(r'Boruta-Pc.csv')
counts = (df.iloc[:, :] <= 1).sum(axis=0).values
M = len(df.columns)
res = estimate_XY_em(counts, 20, n_init=100, max_iter=1000)
print(f'Estimated X: {res["X"]:.4f}, Y: {res["Y"]:.4f}, pi: {res["pi"]:.4f}, log-likelihood: {res["loglik"]:.4f}')
df_ = pd.DataFrame({
    'feature': df.columns,
    'count': counts.astype(int),
    'prob_important': res['resp']
})
df_ = df_[df_['prob_important'] >= 0.95]
df_.reset_index(drop=True, inplace=True)
df_

Estimated X: 0.9182, Y: 0.0010, pi: 0.0519, log-likelihood: -704.6975


,feature,count,prob_important
0,RDKit2D_MinEStateIndex,11,1.0
1,RDKit2D_qed,20,1.0
2,RDKit2D_SPS,9,1.0
3,RDKit2D_BalabanJ,20,1.0
4,RDKit2D_Chi0n,20,1.0
...,...,...,...
65,ChemInfo_ISF,20,1.0
66,ChemInfo_RC1,20,1.0
67,ChemInfo_DIP,20,1.0
68,ChemInfo_DZPE,20,1.0
